# Frente 1 — Limpeza e Tratamento dos Dados

**Responsável:** Artur Alves

# Objetivo

Realizar a exploração inicial, identificação de problemas e limpeza das bases de dados utilizadas no projeto.

In [ ]:
# IMPORTAÇÃO DAS BIBLIOTECAS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# CARREGAMENTO DAS BASES

pedidos = pd.read_csv("../dados/olist_orders_dataset.csv")

itens = pd.read_csv("../dados/olist_order_items_dataset.csv")

produtos = pd.read_csv("../dados/olist_products_dataset.csv")

clientes = pd.read_csv("../dados/olist_customers_dataset.csv")

vendedores = pd.read_csv("../dados/olist_sellers_dataset.csv")

In [ ]:
# JUNÇÃO DAS CINCO BASES

base_completa = itens.merge(
    pedidos,
    on="order_id",
    how="left",
    validate="many_to_one"
)

base_completa = base_completa.merge(
    produtos,
    on="product_id",
    how="left",
    validate="many_to_one"
)

base_completa = base_completa.merge(
    clientes,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

base_completa = base_completa.merge(
    vendedores,
    on="seller_id",
    how="left",
    validate="many_to_one"
)

In [ ]:
# VISUALIZAÇÃO DA BASE COMPLETA

print("Quantidade de linhas e colunas:")
print(base_completa.shape)

print("\nPrimeiras linhas:")
display(base_completa.head())

In [ ]:
# ANÁLISE EXPLORATÓRIA DOS DADOS (EDA)

display(base_completa.describe(include="all").T)

# Valores ausentes
print("\nVALORES AUSENTES:")
display(
    base_completa.isnull().sum()
    .sort_values(ascending=False)
    .loc[lambda x: x > 0]
)

print("\nREGISTROS DUPLICADOS:")
print(base_completa.duplicated().sum())

print("\nTIPOS DE DADOS:")
display(base_completa.dtypes)

In [ ]:
# CONVERSÃO DAS COLUNAS DE DATAS

colunas_data = [
    "shipping_limit_date",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for coluna in colunas_data:
    base_completa[coluna] = pd.to_datetime(
        base_completa[coluna],
        errors="coerce"
    )

display(base_completa[colunas_data].dtypes)

In [ ]:
# VERIFICAÇÃO GERAL DOS DADOS

# Valores ausentes
print("VALORES AUSENTES:")
display(base_completa.isnull().sum().sort_values(ascending=False))

# Situação dos pedidos sem data de entrega
print("\nPEDIDOS SEM DATA DE ENTREGA:")
display(
    base_completa.loc[
        base_completa["order_delivered_customer_date"].isnull(),
        "order_status"
    ].value_counts()
)

# Estatísticas de peso e dimensões
print("\nPESO E DIMENSÕES:")
colunas = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]
display(base_completa[colunas].describe().T)

# Valores iguais a zero
print("\nVALORES IGUAIS A ZERO:")
display((base_completa[colunas] == 0).sum())

In [ ]:
# VERIFICAÇÃO DE PREÇOS E FRETES

print("ESTATÍSTICAS DE PREÇOS:")
display(base_completa["price"].describe())

print("\nESTATÍSTICAS DE FRETES:")
display(base_completa["freight_value"].describe())

print("\nFRETES IGUAIS A ZERO:")
print((base_completa["freight_value"] == 0).sum())

In [47]:
# VERIFICAÇÃO FINAL

print("Quantidade de linhas e colunas:", base_completa.shape)

print("\nRegistros duplicados:", base_completa.duplicated().sum())

print("\nTipos de dados:")
display(base_completa.dtypes)

print("\nQuantidade total de valores ausentes:")
print(base_completa.isnull().sum().sum())

Quantidade de linhas e colunas: (112650, 29)

Registros duplicados: 0

Tipos de dados:


order_id                                    str
order_item_id                             int64
product_id                                  str
seller_id                                   str
shipping_limit_date              datetime64[us]
price                                   float64
freight_value                           float64
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
product_category_name                       str
product_name_lenght                     float64
product_description_lenght              float64
product_photos_qty                      float64
product_weight_g                        float64
product_length_cm                       float64
product_height_cm                       


Quantidade total de valores ausentes:
10147


In [ ]:
# EXPORTAÇÃO DA BASE TRATADA

base_completa.to_csv(
    "../dados/base_tratada.csv",
    index=False,
    encoding="utf-8-sig"
)